In [1]:
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss, MarginRankingLoss
from torch.nn.utils import clip_grad_norm_

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
TRAIN_PATH = Path("corpus/train.jsonl")
VALIDATION_PATH = Path("corpus/validation.jsonl")
TEST_PATH = Path("corpus/test.jsonl")

MODEL_NAME = "neuralmind/bert-large-portuguese-cased"

NON_PUN_LABEL = 0
PUN_LABEL = 1

id2label = {
    0: "0",
    1: "1"
}

label2id = {
    "0": 0,
    "1": 1
}

SEED = 40
MAX_LENGTH = 256
NUM_EPOCHS = 6
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2

ALPHA_PAIR_LOSS = 0.2
PAIR_MARGIN = 0.5

In [3]:
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [4]:
def read_jsonl(file_path):
    rows = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                rows.append(json.loads(line))

    return pd.DataFrame(rows)


train_df = read_jsonl(TRAIN_PATH)
validation_df = read_jsonl(VALIDATION_PATH)
test_df = read_jsonl(TEST_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(validation_df.head())
display(test_df.head())

Train shape: (3990, 5)
Validation shape: (570, 5)
Test shape: (1140, 5)


,id,text,label,tokens,labels
0,5.792.H,Por que a mulher esotérica não conseguia engra...,1,"[Por, que, a, mulher, esotérica, não, consegui...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]"
1,5.733.H,Qual o sambista passou a dar presente pra todo...,1,"[Qual, o, sambista, passou, a, dar, presente, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,4.652.N,Um homem matou uma ovelha e agora foi preso . ...,0,"[Um, homem, matou, uma, ovelha, e, agora, foi,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,5.2585.N,Qual apresentador de TV vive gripado? Fausto S...,0,"[Qual, apresentador, de, TV, vive, gripado, ?,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,5.28.N,Qual é a modelo mais bela que existe? Gisele B...,0,"[Qual, é, a, modelo, mais, bela, que, existe, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


,id,text,label,tokens,labels
0,5.46.H,Por que o carteiro foi à feira? Porque tinha u...,1,"[Por, que, o, carteiro, foi, à, feira, ?, Porq...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]"
1,5.1811.H,Qual é o animal que está sempre cansado? Dorme...,1,"[Qual, é, o, animal, que, está, sempre, cansad...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]"
2,5.1990.N,Uma cerveja se associou a um comediante para a...,0,"[Uma, cerveja, se, associou, a, um, comediante...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,4.103.H,Qual é a única coisa que se faz sempre em nume...,1,"[Qual, é, a, única, coisa, que, se, faz, sempr...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0]"
4,4.19.H,O meu baralho de cartas fica doido quando ligo...,1,"[O, meu, baralho, de, cartas, fica, doido, qua...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


,id,text,label,tokens,labels
0,4.66.N,Eu adoro fiambre . E também queijo.,0,"[Eu, adoro, fiambre, ., E, também, queijo, .]","[0, 0, 0, 0, 0, 0, 0, 0]"
1,5.2933.N,Qual a moeda inacreditável? Euro,0,"[Qual, a, moeda, inacreditável, ?, Euro]","[0, 0, 0, 0, 0, 0]"
2,5.3868.N,Qual marca de hotéis virou presidente? Trump.,0,"[Qual, marca, de, hotéis, virou, presidente, ?...","[0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,5.3612.N,Qual é a loja que fez um estande em volta do d...,0,"[Qual, é, a, loja, que, fez, um, estande, em, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,5.1714.H,Você gosta de Katy Perry? Katy perguntou?,1,"[Você, gosta, de, Katy, Perry, ?, Katy, pergun...","[0, 0, 0, 0, 0, 0, 1, 1, 0]"


In [5]:
required_columns = {"id", "text", "label"}

for split_name, dataframe in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df)
]:
    missing_columns = required_columns - set(dataframe.columns)

    if missing_columns:
        raise ValueError(f"Missing columns in {split_name}: {missing_columns}")

    dataframe["label"] = dataframe["label"].astype(int)

    invalid_labels = set(dataframe["label"].unique()) - {0, 1}

    if invalid_labels:
        raise ValueError(f"Invalid labels in {split_name}: {invalid_labels}")

print("Train label distribution:")
display(train_df["label"].value_counts().sort_index())

print("Validation label distribution:")
display(validation_df["label"].value_counts().sort_index())

print("Test label distribution:")
display(test_df["label"].value_counts().sort_index())

Train label distribution:


,count
label,
0,1995
1,1995


Validation label distribution:


,count
label,
0,285
1,285


Test label distribution:


,count
label,
0,570
1,570


In [6]:
def extract_pair_id(example_id):
    return re.sub(r"\.[HN]$", "", str(example_id))


def extract_pair_suffix(example_id):
    match = re.search(r"\.([HN])$", str(example_id))

    if match:
        return match.group(1)

    return None


for dataframe in [train_df, validation_df, test_df]:
    dataframe["pair_id"] = dataframe["id"].apply(extract_pair_id)
    dataframe["pair_suffix"] = dataframe["id"].apply(extract_pair_suffix)


print("Train suffix vs label:")
display(pd.crosstab(train_df["pair_suffix"], train_df["label"]))

print("Validation suffix vs label:")
display(pd.crosstab(validation_df["pair_suffix"], validation_df["label"]))

print("Test suffix vs label:")
display(pd.crosstab(test_df["pair_suffix"], test_df["label"]))

Train suffix vs label:


label,0,1
pair_suffix,,
H,0,1995
N,1995,0


Validation suffix vs label:


label,0,1
pair_suffix,,
H,0,285
N,285,0


Test suffix vs label:


label,0,1
pair_suffix,,
H,0,570
N,570,0


In [7]:
def validate_complete_pairs(dataframe, split_name):
    pair_sizes = dataframe.groupby("pair_id").size()
    invalid_size_pairs = pair_sizes[pair_sizes != 2]

    if len(invalid_size_pairs) > 0:
        display(invalid_size_pairs.head(20))
        raise ValueError(
            f"{split_name}: {len(invalid_size_pairs)} pairs do not have exactly 2 examples."
        )

    invalid_label_pairs = []

    for pair_id, group in dataframe.groupby("pair_id"):
        labels = set(group["label"].tolist())

        if labels != {NON_PUN_LABEL, PUN_LABEL}:
            invalid_label_pairs.append(pair_id)

    if len(invalid_label_pairs) > 0:
        raise ValueError(
            f"{split_name}: {len(invalid_label_pairs)} pairs do not have exactly one 0 and one 1."
        )

    print(f"{split_name}: valid pairs.")
    print(f"{split_name}: {len(pair_sizes)} pairs, {len(dataframe)} examples.")


def validate_suffix_label_consistency(dataframe, split_name):
    invalid_rows = dataframe[
        ((dataframe["pair_suffix"] == "H") & (dataframe["label"] != PUN_LABEL)) |
        ((dataframe["pair_suffix"] == "N") & (dataframe["label"] != NON_PUN_LABEL)) |
        (dataframe["pair_suffix"].isna())
    ]

    if len(invalid_rows) > 0:
        display(invalid_rows.head(20))
        raise ValueError(
            f"{split_name}: existem exemplos em que o sufixo .H/.N "
            f"não corresponde à label esperada."
        )

    print(f"{split_name}: suffix-label consistency OK.")


validate_complete_pairs(train_df, "Train")
validate_complete_pairs(validation_df, "Validation")
validate_complete_pairs(test_df, "Test")

validate_suffix_label_consistency(train_df, "Train")
validate_suffix_label_consistency(validation_df, "Validation")
validate_suffix_label_consistency(test_df, "Test")

Train: valid pairs.
Train: 1995 pairs, 3990 examples.
Validation: valid pairs.
Validation: 285 pairs, 570 examples.
Test: valid pairs.
Test: 570 pairs, 1140 examples.
Train: suffix-label consistency OK.
Validation: suffix-label consistency OK.
Test: suffix-label consistency OK.


In [8]:
train_pair_ids = set(train_df["pair_id"])
validation_pair_ids = set(validation_df["pair_id"])
test_pair_ids = set(test_df["pair_id"])

train_validation_overlap = train_pair_ids.intersection(validation_pair_ids)
train_test_overlap = train_pair_ids.intersection(test_pair_ids)
validation_test_overlap = validation_pair_ids.intersection(test_pair_ids)

print("Train/validation pair overlap:", len(train_validation_overlap))
print("Train/test pair overlap:", len(train_test_overlap))
print("Validation/test pair overlap:", len(validation_test_overlap))

if train_validation_overlap or train_test_overlap or validation_test_overlap:
    raise ValueError(
        "The reorganized corpus still has pair_id overlap between splits."
    )

print("OK: no pair_id overlap between train, validation and test.")

Train/validation pair overlap: 0
Train/test pair overlap: 0
Validation/test pair overlap: 0
OK: no pair_id overlap between train, validation and test.


In [9]:
def build_pair_dataframe(dataframe):
    pair_rows = []

    for pair_id, group in dataframe.groupby("pair_id"):
        pun_row = group[group["label"] == PUN_LABEL].iloc[0]
        non_pun_row = group[group["label"] == NON_PUN_LABEL].iloc[0]

        pair_rows.append({
            "pair_id": pair_id,

            "pun_id": pun_row["id"],
            "pun_text": pun_row["text"],

            "non_pun_id": non_pun_row["id"],
            "non_pun_text": non_pun_row["text"]
        })

    return pd.DataFrame(pair_rows)


train_pairs_df = build_pair_dataframe(train_df)
validation_pairs_df = build_pair_dataframe(validation_df)
test_pairs_df = build_pair_dataframe(test_df)

print("Train pairs:", train_pairs_df.shape)
print("Validation pairs:", validation_pairs_df.shape)
print("Test pairs:", test_pairs_df.shape)

display(train_pairs_df.head())

Train pairs: (1995, 5)
Validation pairs: (285, 5)
Test pairs: (570, 5)


,pair_id,pun_id,pun_text,non_pun_id,non_pun_text
0,1.1,1.1.H,Deve ser difícil ser professor de natação. Voc...,1.1.N,Deve ser difícil ser professor de natação . Vo...
1,1.4,1.4.H,O que uma impressora falou para a outra? Essa ...,1.4.N,O que uma impressora falou para a outra? Essa ...
2,1.5,1.5.H,Por que a galinha bateu a cabeça contra a pare...,1.5.N,Por que a galinha bateu a cabeça contra a pare...
3,2.10,2.10.H,Como o padre bateu o carro? Dando uma rezinha.,2.10.N,Como o padre bateu o carro? Dando uma ré.
4,2.12,2.12.H,Por que a vaca foi para o espaço? Para se enco...,2.12.N,Por que a vaca foi para o espaço? Para se enco...


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [11]:
class PairedPunDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        return {
            "pair_id": row["pair_id"],

            "pun_id": row["pun_id"],
            "pun_text": row["pun_text"],

            "non_pun_id": row["non_pun_id"],
            "non_pun_text": row["non_pun_text"]
        }


class SingleTextDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        return {
            "id": row["id"],
            "pair_id": row["pair_id"],
            "text": row["text"],
            "label": int(row["label"])
        }

In [12]:
def paired_collate_fn(batch):
    pun_texts = [item["pun_text"] for item in batch]
    non_pun_texts = [item["non_pun_text"] for item in batch]

    pun_encoding = tokenizer(
        pun_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    non_pun_encoding = tokenizer(
        non_pun_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    return {
        "pair_ids": [item["pair_id"] for item in batch],

        "pun_ids": [item["pun_id"] for item in batch],
        "non_pun_ids": [item["non_pun_id"] for item in batch],

        "pun_encoding": pun_encoding,
        "non_pun_encoding": non_pun_encoding
    }


def single_collate_fn(batch):
    texts = [item["text"] for item in batch]

    encoding = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long
    )

    return {
        "ids": [item["id"] for item in batch],
        "pair_ids": [item["pair_id"] for item in batch],
        "encoding": encoding,
        "labels": labels
    }

In [13]:
train_pair_dataset = PairedPunDataset(train_pairs_df)

validation_single_dataset = SingleTextDataset(validation_df)
test_single_dataset = SingleTextDataset(test_df)

PAIR_BATCH_SIZE = max(1, TRAIN_BATCH_SIZE // 2)

train_pair_loader = DataLoader(
    train_pair_dataset,
    batch_size=PAIR_BATCH_SIZE,
    shuffle=True,
    collate_fn=paired_collate_fn
)

print(
    f"Train pair batch size: {PAIR_BATCH_SIZE} pairs "
    f"= {PAIR_BATCH_SIZE * 2} texts per update"
)

validation_single_loader = DataLoader(
    validation_single_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=single_collate_fn
)

test_single_loader = DataLoader(
    test_single_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=single_collate_fn
)

print("Train pair batches:", len(train_pair_loader))
print("Validation batches:", len(validation_single_loader))
print("Test batches:", len(test_single_loader))

Train pair batch size: 4 pairs = 8 texts per update
Train pair batches: 499
Validation batches: 72
Test batches: 143


In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_training_steps = len(train_pair_loader) * NUM_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

classification_loss_fn = CrossEntropyLoss()
pair_ranking_loss_fn = MarginRankingLoss(margin=PAIR_MARGIN)

use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

print("Total training steps:", total_training_steps)
print("Warmup steps:", warmup_steps)
print("Mixed precision:", use_amp)

pytorch_model.bin:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

Total training steps: 2994
Warmup steps: 299
Mixed precision: True


/tmp/ipykernel_2199/1260460783.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


In [15]:
def compute_basic_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),

        "precision_macro": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "recall_macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "precision_weighted": precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "recall_weighted": recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "f1_weighted": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
    }


def compute_pair_metrics(predictions_df):
    pair_exact_results = []
    pair_ranking_results = []

    for pair_id, group in predictions_df.groupby("pair_id"):
        if len(group) != 2:
            continue

        pair_exact_correct = bool(
            (group["label"] == group["prediction"]).all()
        )

        pair_exact_results.append(pair_exact_correct)

        pun_rows = group[group["label"] == PUN_LABEL]
        non_pun_rows = group[group["label"] == NON_PUN_LABEL]

        if len(pun_rows) == 1 and len(non_pun_rows) == 1:
            pun_score = float(pun_rows.iloc[0]["score_pun"])
            non_pun_score = float(non_pun_rows.iloc[0]["score_pun"])

            pair_ranking_correct = pun_score > non_pun_score
            pair_ranking_results.append(pair_ranking_correct)

    return {
        "pair_exact_accuracy": float(np.mean(pair_exact_results)) if pair_exact_results else 0.0,
        "pair_ranking_accuracy": float(np.mean(pair_ranking_results)) if pair_ranking_results else 0.0,
        "evaluated_pairs": int(len(pair_exact_results))
    }


def evaluate_model(model, dataloader, threshold=0.0):
    model.eval()

    all_ids = []
    all_pair_ids = []
    all_labels = []
    all_predictions = []
    all_scores = []
    all_prob_0 = []
    all_prob_1 = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            encoding = {
                key: value.to(device)
                for key, value in batch["encoding"].items()
            }

            labels = batch["labels"].to(device)

            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(**encoding)
                logits = outputs.logits

            probabilities = torch.softmax(logits, dim=1)

            scores = logits[:, PUN_LABEL] - logits[:, NON_PUN_LABEL]

            predictions = (scores > threshold).long()

            all_ids.extend(batch["ids"])
            all_pair_ids.extend(batch["pair_ids"])
            all_labels.extend(labels.cpu().numpy().tolist())
            all_predictions.extend(predictions.cpu().numpy().tolist())
            all_scores.extend(scores.cpu().numpy().tolist())
            all_prob_0.extend(probabilities[:, NON_PUN_LABEL].cpu().numpy().tolist())
            all_prob_1.extend(probabilities[:, PUN_LABEL].cpu().numpy().tolist())

    predictions_df = pd.DataFrame({
        "id": all_ids,
        "pair_id": all_pair_ids,
        "label": all_labels,
        "prediction": all_predictions,
        "score_pun": all_scores,
        "prob_0": all_prob_0,
        "prob_1": all_prob_1
    })

    metrics = compute_basic_metrics(
        predictions_df["label"].tolist(),
        predictions_df["prediction"].tolist()
    )

    metrics.update(compute_pair_metrics(predictions_df))

    return metrics, predictions_df

In [16]:
def find_best_threshold(predictions_df, metric_name="accuracy"):
    thresholds = np.linspace(
        predictions_df["score_pun"].min(),
        predictions_df["score_pun"].max(),
        200
    )

    best_threshold = 0.0
    best_value = -1.0

    y_true = predictions_df["label"].to_numpy()
    scores = predictions_df["score_pun"].to_numpy()

    for threshold in thresholds:
        y_pred = (scores > threshold).astype(int)

        if metric_name == "accuracy":
            value = accuracy_score(y_true, y_pred)
        elif metric_name == "f1_macro":
            value = f1_score(y_true, y_pred, average="macro", zero_division=0)
        else:
            raise ValueError("metric_name must be 'accuracy' or 'f1_macro'.")

        if value > best_value:
            best_value = value
            best_threshold = threshold

    return best_threshold, best_value

In [17]:
def apply_pairwise_decoding(predictions_df):
    decoded_df = predictions_df.copy()

    decoded_df["prediction_threshold"] = decoded_df["prediction"]

    for pair_id, group in decoded_df.groupby("pair_id"):
        if len(group) != 2:
            continue

        sorted_indices = group.sort_values(
            "score_pun",
            ascending=False
        ).index.tolist()

        pun_index = sorted_indices[0]
        non_pun_index = sorted_indices[1]

        decoded_df.loc[pun_index, "prediction"] = PUN_LABEL
        decoded_df.loc[non_pun_index, "prediction"] = NON_PUN_LABEL

    return decoded_df

In [18]:
PRIMARY_THRESHOLD = 0.0

best_validation_f1 = -1.0
best_validation_accuracy_at_best_f1 = -1.0
best_validation_threshold = PRIMARY_THRESHOLD

best_model_state = None
epochs_without_improvement = 0

training_history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()

    total_loss = 0.0
    total_classification_loss = 0.0
    total_pair_loss = 0.0

    progress_bar = tqdm(
        train_pair_loader,
        desc=f"Epoch {epoch}/{NUM_EPOCHS}",
        leave=True
    )

    for batch in progress_bar:
        optimizer.zero_grad()

        pun_encoding = {
            key: value.to(device)
            for key, value in batch["pun_encoding"].items()
        }

        non_pun_encoding = {
            key: value.to(device)
            for key, value in batch["non_pun_encoding"].items()
        }

        current_batch_size = pun_encoding["input_ids"].size(0)

        with torch.cuda.amp.autocast(enabled=use_amp):
            pun_outputs = model(**pun_encoding)
            non_pun_outputs = model(**non_pun_encoding)

            pun_logits = pun_outputs.logits
            non_pun_logits = non_pun_outputs.logits

            all_logits = torch.cat(
                [pun_logits, non_pun_logits],
                dim=0
            )

            all_labels = torch.cat(
                [
                    torch.full(
                        (current_batch_size,),
                        PUN_LABEL,
                        dtype=torch.long,
                        device=device
                    ),
                    torch.full(
                        (current_batch_size,),
                        NON_PUN_LABEL,
                        dtype=torch.long,
                        device=device
                    )
                ],
                dim=0
            )

            classification_loss = classification_loss_fn(
                all_logits,
                all_labels
            )

            pun_scores = pun_logits[:, PUN_LABEL] - pun_logits[:, NON_PUN_LABEL]
            non_pun_scores = non_pun_logits[:, PUN_LABEL] - non_pun_logits[:, NON_PUN_LABEL]

            ranking_targets = torch.ones_like(pun_scores)

            pair_loss = pair_ranking_loss_fn(
                pun_scores,
                non_pun_scores,
                ranking_targets
            )

            loss = classification_loss + ALPHA_PAIR_LOSS * pair_loss

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        total_loss += loss.item()
        total_classification_loss += classification_loss.item()
        total_pair_loss += pair_loss.item()

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "ce": f"{classification_loss.item():.4f}",
            "pair": f"{pair_loss.item():.4f}"
        })

    average_loss = total_loss / len(train_pair_loader)
    average_classification_loss = total_classification_loss / len(train_pair_loader)
    average_pair_loss = total_pair_loss / len(train_pair_loader)

    validation_metrics, validation_predictions_df = evaluate_model(
        model=model,
        dataloader=validation_single_loader,
        threshold=PRIMARY_THRESHOLD
    )

    current_validation_accuracy = validation_metrics["accuracy"]
    current_validation_f1 = validation_metrics["f1_macro"]

    aux_epoch_threshold, aux_epoch_f1 = find_best_threshold(
        predictions_df=validation_predictions_df,
        metric_name="f1_macro"
    )

    aux_validation_metrics, _ = evaluate_model(
        model=model,
        dataloader=validation_single_loader,
        threshold=aux_epoch_threshold
    )

    epoch_record = {
        "epoch": epoch,
        "train_loss": average_loss,
        "train_classification_loss": average_classification_loss,
        "train_pair_loss": average_pair_loss,

        "primary_threshold": PRIMARY_THRESHOLD,
        "aux_best_threshold": aux_epoch_threshold,
        "aux_validation_accuracy": aux_validation_metrics["accuracy"],
        "aux_validation_f1_macro": aux_validation_metrics["f1_macro"],

        **{f"validation_{key}": value for key, value in validation_metrics.items()}
    }

    training_history.append(epoch_record)

    print("\nEpoch summary:")
    for key, value in epoch_record.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

    if current_validation_f1 > best_validation_f1:
        best_validation_f1 = current_validation_f1
        best_validation_accuracy_at_best_f1 = current_validation_accuracy
        best_validation_threshold = PRIMARY_THRESHOLD
        epochs_without_improvement = 0

        best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

        print(f"New best validation f1_macro: {best_validation_f1:.4f}")
        print(
            f"Validation accuracy at best f1_macro: "
            f"{best_validation_accuracy_at_best_f1:.4f}"
        )

    else:
        epochs_without_improvement += 1

        print(
            f"No improvement. "
            f"Patience: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}"
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break


history_df = pd.DataFrame(training_history)

print("Training history:")
display(history_df)

Epoch 1/6:   0%|          | 0/499 [00:00<?, ?it/s]

/tmp/ipykernel_2199/3695176291.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

/tmp/ipykernel_2199/1593722181.py:100: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

...

/tmp/ipykernel_2199/1593722181.py:100: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

/tmp/ipykernel_2199/1593722181.py:100: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):



Epoch summary:
epoch: 6
train_loss: 0.1448
train_classification_loss: 0.1388
train_pair_loss: 0.0298
primary_threshold: 0.0000
aux_best_threshold: -4.2345
aux_validation_accuracy: 0.7789
aux_validation_f1_macro: 0.7789
validation_accuracy: 0.7649
validation_precision_macro: 0.7683
validation_recall_macro: 0.7649
validation_f1_macro: 0.7642
validation_precision_weighted: 0.7683
validation_recall_weighted: 0.7649
validation_f1_weighted: 0.7642
validation_pair_exact_accuracy: 0.5509
validation_pair_ranking_accuracy: 0.8526
validation_evaluated_pairs: 285
No improvement. Patience: 2/2
Early stopping triggered.
Training history:


,epoch,train_loss,train_classification_loss,train_pair_loss,primary_threshold,aux_best_threshold,aux_validation_accuracy,aux_validation_f1_macro,validation_accuracy,validation_precision_macro,validation_recall_macro,validation_f1_macro,validation_precision_weighted,validation_recall_weighted,validation_f1_weighted,validation_pair_exact_accuracy,validation_pair_ranking_accuracy,validation_evaluated_pairs
0,1,0.675745,0.613334,0.312054,0.0,-0.801213,0.764912,0.764909,0.701754,0.734144,0.701754,0.691071,0.734144,0.701754,0.691071,0.431579,0.845614,285
1,2,0.478876,0.449270,0.148029,0.0,-0.078302,0.768421,0.768375,0.764912,0.765383,0.764912,0.764808,0.765383,0.764912,0.764808,0.554386,0.873684,285
2,3,0.354249,0.342625,0.058123,0.0,-2.595556,0.773684,0.773678,0.757895,0.761618,0.757895,0.757030,0.761618,0.757895,0.757030,0.540351,0.845614,285
3,4,0.289778,0.281683,0.040475,0.0,-0.999254,0.780702,0.780669,0.773684,0.774444,0.773684,0.773527,0.774444,0.773684,0.773527,0.571930,0.877193,285
4,5,0.189802,0.184650,0.025757,0.0,-4.601680,0.780702,0.780701,0.754386,0.761812,0.754386,0.752632,0.761812,0.754386,0.752632,0.536842,0.863158,285
5,6,0.144767,0.138815,0.029761,0.0,-4.234454,0.778947,0.778945,0.764912,0.768295,0.764912,0.764169,0.768295,0.764912,0.764169,0.550877,0.852632,285


In [19]:
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model.to(device)

    print(f"Best validation f1_macro restored: {best_validation_f1:.4f}")
    print(
        f"Validation accuracy at best f1_macro restored: "
        f"{best_validation_accuracy_at_best_f1:.4f}"
    )
    print(f"Primary threshold restored: {best_validation_threshold:.4f}")
else:
    print("No best model state was stored.")

Best validation f1_macro restored: 0.7735
Validation accuracy at best f1_macro restored: 0.7737
Primary threshold restored: 0.0000


In [20]:
test_metrics, test_predictions_df = evaluate_model(
    model=model,
    dataloader=test_single_loader,
    threshold=best_validation_threshold
)

print("=== MAIN TEST RESULTS: standard binary classification ===")

for metric_name, metric_value in test_metrics.items():
    if isinstance(metric_value, float):
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value}")

y_true = test_predictions_df["label"].tolist()
y_pred = test_predictions_df["prediction"].tolist()

print("\n=== MAIN CLASSIFICATION REPORT: standard binary classification ===")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["0", "1"],
        digits=2,
        zero_division=0
    )
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=["true_0", "true_1"],
    columns=["pred_0", "pred_1"]
)

display(cm_df)

print("\n=== AUXILIARY PAIR ANALYSIS: without changing predictions ===")
print(f"Pair exact accuracy: {test_metrics['pair_exact_accuracy']:.4f}")
print(f"Pair ranking accuracy: {test_metrics['pair_ranking_accuracy']:.4f}")
print(f"Evaluated pairs: {test_metrics['evaluated_pairs']}")

Evaluating:   0%|          | 0/143 [00:00<?, ?it/s]

/tmp/ipykernel_2199/1593722181.py:100: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


=== MAIN TEST RESULTS: standard binary classification ===
accuracy: 0.7649
precision_macro: 0.7652
recall_macro: 0.7649
f1_macro: 0.7648
precision_weighted: 0.7652
recall_weighted: 0.7649
f1_weighted: 0.7648
pair_exact_accuracy: 0.5544
pair_ranking_accuracy: 0.8491
evaluated_pairs: 570

=== MAIN CLASSIFICATION REPORT: standard binary classification ===
              precision    recall  f1-score   support

           0       0.76      0.78      0.77       570
           1       0.77      0.75      0.76       570

    accuracy                           0.76      1140
   macro avg       0.77      0.76      0.76      1140
weighted avg       0.77      0.76      0.76      1140



,pred_0,pred_1
true_0,446,124
true_1,144,426



=== AUXILIARY PAIR ANALYSIS: without changing predictions ===
Pair exact accuracy: 0.5544
Pair ranking accuracy: 0.8491
Evaluated pairs: 570


In [21]:
pairwise_test_predictions_df = apply_pairwise_decoding(
    test_predictions_df
)

pairwise_test_metrics = compute_basic_metrics(
    pairwise_test_predictions_df["label"].tolist(),
    pairwise_test_predictions_df["prediction"].tolist()
)

pairwise_test_metrics.update(
    compute_pair_metrics(pairwise_test_predictions_df)
)

print("\n=== AUXILIARY ONLY: pairwise decoding results ===")
print(
    "Atenção: este resultado força cada par a ter uma previsão 1 e uma previsão 0. "
    "Use apenas como análise auxiliar, não como resultado principal."
)

for metric_name, metric_value in pairwise_test_metrics.items():
    if isinstance(metric_value, float):
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value}")

pairwise_y_true = pairwise_test_predictions_df["label"].tolist()
pairwise_y_pred = pairwise_test_predictions_df["prediction"].tolist()

print("\n=== AUXILIARY CLASSIFICATION REPORT: pairwise decoding ===")
print(
    classification_report(
        pairwise_y_true,
        pairwise_y_pred,
        labels=[0, 1],
        target_names=["0", "1"],
        digits=2,
        zero_division=0
    )
)

pairwise_cm = confusion_matrix(
    pairwise_y_true,
    pairwise_y_pred,
    labels=[0, 1]
)

pairwise_cm_df = pd.DataFrame(
    pairwise_cm,
    index=["true_0", "true_1"],
    columns=["pred_0", "pred_1"]
)

display(pairwise_cm_df)


=== AUXILIARY ONLY: pairwise decoding results ===
Atenção: este resultado força cada par a ter uma previsão 1 e uma previsão 0. Use apenas como análise auxiliar, não como resultado principal.
accuracy: 0.8491
precision_macro: 0.8491
recall_macro: 0.8491
f1_macro: 0.8491
precision_weighted: 0.8491
recall_weighted: 0.8491
f1_weighted: 0.8491
pair_exact_accuracy: 0.8491
pair_ranking_accuracy: 0.8491
evaluated_pairs: 570

=== AUXILIARY CLASSIFICATION REPORT: pairwise decoding ===
              precision    recall  f1-score   support

           0       0.85      0.85      0.85       570
           1       0.85      0.85      0.85       570

    accuracy                           0.85      1140
   macro avg       0.85      0.85      0.85      1140
weighted avg       0.85      0.85      0.85      1140



,pred_0,pred_1
true_0,484,86
true_1,86,484


In [22]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=["true_0", "true_1"],
    columns=["pred_0", "pred_1"]
)

display(cm_df)

print("=== Pair-based metrics with pairwise decoding ===")
print(f"Pair exact accuracy: {pairwise_test_metrics['pair_exact_accuracy']:.4f}")
print(f"Pair ranking accuracy: {pairwise_test_metrics['pair_ranking_accuracy']:.4f}")
print(f"Evaluated pairs: {pairwise_test_metrics['evaluated_pairs']}")

,pred_0,pred_1
true_0,446,124
true_1,144,426


=== Pair-based metrics with pairwise decoding ===
Pair exact accuracy: 0.8491
Pair ranking accuracy: 0.8491
Evaluated pairs: 570
